# Adding Sources to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Source components**. Sources represent components that **produce a commodity and inject it into the energy system**. We focus on the most essential parameters required to define and understand a Source component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of sources include:

- renewable generation technologies such as wind turbines or photovoltaic systems
- fossil fuel extraction
- imports of commodities such as natural gas or electricity
- external supply such as hydrogen delivery

Sources therefore represent **entry points of commodities into the modeled system**.

The Sink component in FINE is closely related to the Source component and internally inherits from the same class. Therefore, the parameters described here are also relevant when modeling sinks.


## Initialize an energy system model

Before we can add sources, we need to initialize the energy system model as shown in the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb).

In [ ]:
import fine as fn  # Provides objects and functions to model an energy system
import pandas as pd  # Used to manage data in tables
import numpy as np  # Used to generate random input data
np.random.seed(42)  # Sets a "seed" to produce the same random input data in each model run

esM = fn.EnergySystemModel(
    locations = {"regionN", "regionS"},
    commodities = {"electricity", "naturalGas", "CO2"},
    commodityUnitsDict = {
    "electricity": r"GW$_{el}$",
    "naturalGas": r"GW$_{CH_{4},LHV}$",
    "CO2": r"Mio. t$_{CO_2}$/h",
    },
    costUnit = "1e6 Euro",
    lengthUnit = "km",
    numberOfTimeSteps = 8760,
    hoursPerTimeStep = 1
)

## Add Sources

### Wind

We can now add wind turbines as a first source. Below you can find a more detailed explanation of the parameters used here.

In [ ]:
esM.add(
    fn.Source(
        esM = esM,
        name = "Wind turbines", # Name of the source
        commodity = "electricity", # Name of the commodity produced by the source, as set during initialization
        hasCapacityVariable = True, # Specifies whether the source has a capacity variable
        operationRateMax = pd.DataFrame(
            [[np.random.beta(a=2, b=7.5), np.random.beta(a=2, b=9)] for t in range(8760)],
            index=range(8760),
            columns=["regionN", "regionS"]
            ).round(6), # Defines the maximum operation rate for each time step and location. Was set here randomly.
        capacityMax = pd.Series([400, 200], index=["regionN", "regionS"]), # Indicates the maximum capacity of this source for each location
        investPerCapacity = 1200, # Describes the investment costs for one unit of the capacity
        opexPerCapacity = 24, # Describes the operational cost for one unit of capacity
        interestRate = 0.08, # Describes the interest rate which is considered for computing the annuities of the invest of the component (depreciates the invests over the economic lifetime)
        economicLifetime = 20, # Describes the economic lifetime of the component which is considered for computing the annuities of the invest of the component (i.e. the depreciation time)
    )
)

### Solar PV

Next, we can add solar PV as a source. To simulate the time-dependent potential output of electricity generated by solar PV, we assign a daily profile to the maximum operation rate. This is, of course, a highly simplified approach.

In [ ]:
dailyProfileSimple = [ # hourly intensity of solar radiation per day
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0.05,
    0.15,
    0.2,
    0.4,
    0.8,
    0.7,
    0.4,
    0.2,
    0.15,
    0.05,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
]

esM.add(
    fn.Source(
        esM = esM,
        name = "PV",
        commodity = "electricity",
        hasCapacityVariable = True,
        operationRateMax = pd.DataFrame(
            [[u, u] for day in range(365) for u in dailyProfileSimple],
            index=range(8760),
            columns=["regionN", "regionS"],
            ), # daily profile is added to each day and equally in both locations
        capacityMax = pd.Series([100, 100], index=["regionN", "regionS"]), 
        investPerCapacity = 800,
        opexPerCapacity = 16,
        interestRate = 0.8,
        economicLifetime = 25,
    )
)

## General Structure of a Source Instance

The following code snippet shows the structure of the `Source` class and its arguments.

```python
fn.Source(
    esM,
    name,
    commodity,
    hasCapacityVariable,
    capacityVariableDomain="continuous",
    capacityPerPlantUnit=1,
    hasIsBuiltBinaryVariable=False,
    bigM=None,
    operationRateMin=None,
    operationRateMax=None,
    operationRateFix=None,
    tsaWeight=1,
    locationalEligibility=None,
    capacityMin=None,
    capacityMax=None,
    partLoadMin=None,
    sharedPotentialID=None,
    linkedQuantityID=None,
    capacityFix=None,
    commissioningMin=None,
    commissioningMax=None,
    commissioningFix=None,
    isBuiltFix=None,
    investPerCapacity=0,
    investIfBuilt=0,
    opexPerOperation=0,
    commodityCost=0,
    commodityRevenue=0,
    commodityCostTimeSeries=None,
    commodityRevenueTimeSeries=None,
    opexPerCapacity=0,
    opexIfBuilt=0,
    QPcostScale=0,
    interestRate=0.08,
    economicLifetime=10,
    technicalLifetime=None,
    yearlyFullLoadHoursMin=None,
    yearlyFullLoadHoursMax=None,
    balanceLimitID=None,
    pathwayBalanceLimitID=None,
    stockCommissioning=None,
    floorTechnicalLifetime=True,
    pwlcfParameters=None,
    )
```
In the following sections, we explain the most important arguments of a Source component.


## Required Arguments

### esM

`esM` is the energy system model to which the source is added.

### name

`name` is a string, which should describe the type of source which is added to the energy system model.

Examples:
- "Wind turbines"
- "PV"
- "Natural gas import"

### commodity

`commodity` defines which commodity the source produces.

The commodity must be one of the commodities that were defined when initializing the `EnergySystemModel`.

Examples of commodities produced by sources include:<br>
- electricity from renewable generation
- hydrogen from external supply
- natural gas imports

### hasCapacityVariable

`hasCapacityVariable` is a **boolean**, which specifies whether the component has a capacity limit.

Examples:<br>
- A wind turbine has a capacity given in GW_electric -> ```hasCapacityVariable = True```
- Emitting CO2 into the environment is not per se limited by a capacity -> ```hasCapacityVariable = False```

## Optional Parameters

### operationRateMax

`operationRateMax` defines the maximum operation rate for each time step and location. It depends on `hasCapacityVariable` as follows:

- If ```hasCapacityVariable = True```, the values are given relative to the installed capacities (i.e. a value of 1 indicates a utilization of 100% of the capacity).
- If ```hasCapacityVariable = False```, the values are given as absolute values in form of the `commodityUnit` for each time step.

Type:

- None (default)
- Pandas DataFrame with positive (>= 0) entries. The row indices have to match the in the energy system model specified time steps. The column indices have to equal the in the energy system model specified locations. The data in ineligible locations are set to zero.
- Dictionary with investment periods as keys and one of the two options above as values

### commodityCost

`commodityCost` describes the cost value of one operation's unit of the component. The cost unit in which the parameter is given has to match the one specified in the energy system model. 
The total cost is calculated as
$$
\text{commodityCost} \times \text{total annual operation.}
$$

Example: In a national energy system, natural gas could be purchased from another country with a certain cost.

Type: 
- positive float ($\geq 0$)
- Pandas Series with positive floats ($\geq 0$). The indices of the series have to equal the locations as specified in the energy system model.
- Dictionary with investment periods as keys and one of the two options above as values.

### capacityMax

`capacityMax` indicates the maximum capacity of this source.

Example: 

Type:
- None (default)
- float
- int
- Pandas Series with positive values ($\geq 0$). The indices of the series have to equal the in the energy system model specified locations (dimension=1dim) or connections between these locations in the format of 'loc1' + '_' + 'loc2' (dimension=2dim).
- Pandas DataFrame with positive values ($\geq 0$). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.
- Dictionary with investment periods as keys and one of the options above as values.

### investPerCapacity

`investPerCapacity` describes the investment costs for one unit of the capacity. The invest of a component is obtained by multiplying the commissioned capacities of the component (in the physical Unit of the component) with the `investPerCapacity` factor and is distributed over the components technical lifetime. The value has to match the unit costUnit/physicalUnit (e.g. Euro/kW).

Example:

Type:
- float or Pandas Series with location specific values (dimension=1dim). The cost unit in which the parameter is given has to match the one specified in the energy system model (e.g. Euro, Dollar, 1e6 Euro). The value has to match the unit 
$$
\frac{\text{costUnit}}{\text{physicalUnit}}, \qquad \text{e.g.} \quad \frac{\text{Euro}}{\text{kW}}\text{, } \frac{\text{1e6 Euro}}{\text{GW}}.
$$
- float or Pandas Series or DataFrame with location specific values (dimension=2dim). The cost unit in which the parameter is given has to match the one specified in the energy system model divided by the specified lengthUnit (e.g. Euro/m, Dollar/m, 1e6 Euro/km). The value has to match the unit 
$$
\frac{\text{costUnit}}{\text{lengthUnit} \cdot \text{physicalUnit}}, \qquad \text{e.g.} \quad \frac{\text{Euro}}{\text{kW} \cdot \text{m}}\text{, } \frac{\text{1e6 Euro}}{\text{GW} \cdot \text{km}}.
$$
- Dictionary with years as keys (past years which had stock commissioning and investment periods which will be optimized) and one of the two options above as values, e.g. ```{2020: 1000, 2025: 800, 2030: 750}```

The default value is 0.

### opexPerCapacity

`opexPerCapacity` describes the operational cost for one unit of capacity. The annual operational cost, which are only a function of the capacity of the component (in the physicalUnit of the component) and not of the specific operation itself, are obtained by multiplying the commissioned capacity of the component at a location with the `opexPerCapacity` factor and is distributed over the components technical lifetime. The possible types are similar to the ones for [investPerCapacity](#investpercapacity).


### interestRate

`interestRate` describes the interest rate which is considered for computing the annuities of the invest of the component (depreciates the invests over the economic lifetime). A value of 0.08 corresponds to an interest rate of 8%. The interest rate is currently constant for all investment periods.
Warning: The interest must be greater than 0 if [annuityPerpetuity](../_01_initialize/_1_initialize_ESM.ipynb#annuityPerpetuity) is used in the energy system model.

Type:
- None
- Pandas Series with positive values ($\geq 0$). The indices of the series have to equal the in the energy system model specified locations (dimension=1dim) or connections between these locations in the format of 'loc1' + '_' + 'loc2' (dimension=2dim)
- Pandas DataFrame with positive values ($\geq 0$). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.

### economicLifetime

`economicLifetime` describes the economic lifetime of the component which is considered for computing the annuities of the invest of the component (i.e. the depreciation time). The economic lifetime is currently constant over the pathway of investment periods.

Type:
- None
- Pandas Series with positive values ($\geq 0$). The indices of the series have to equal the in the energy system model specified locations (dimension=1dim) or connections between these locations in the format of 'loc1' + '_' + 'loc2' (dimension=2dim)
- Pandas DataFrame with positive ($\geq 0$) values. The row and column indices of the DataFrame have to equal the in the energy system model specified locations.

Many parameters were left out here. Some of them might need a page on their own. Others could be collected in an "other features" notebook